# Crop Disease Detection — EfficientNet-B0 Training

**Runtime**: GPU (T4 recommended) — Runtime → Change runtime type → T4 GPU  
**Dataset**: PlantVillage (38 classes, ~54k images) — preprocessed splits in S3 
**Model**: EfficientNet-B0 fine-tuned with PyTorch  
**Tracking**: MLflow (remote server on EC2)  

## Workflow
1. Install dependencies
2. Download from S3
3. Build DataLoaders with augmentation
4. Fine-tune EfficientNet-B0
5. Evaluate on test set
6. Export to ONNX
7. Log everything to MLflow and register model

## 0. Setup

In [ ]:
# Install dependencies not available in Colab by default
!pip install mlflow boto3 --quiet

In [ ]:
import json
import os
import time
from pathlib import Path

import mlflow
import mlflow.pytorch
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, f1_score
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

# verify GPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Configuration

Edit the values in this cell before running.

In [ ]:
# ── MLflow ────────────────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI = "http://YOUR_EC2_PUBLIC_DNS:5000"  # <- set your EC2 URI
EXPERIMENT_NAME = "crop-disease-detection"
MODEL_NAME = "crop-disease-efficientnet-b0"  # name in MLflow Model Registry

# ── Data ──────────────────────────────────────────────────────────────────────
# S3
S3_BUCKET = "your-model-bucket"
S3_DATA_PREFIX = "data/processed"
LOCAL_DATA_DIR = Path("/content/data/processed")

# ── Training hyperparameters ───────────────────────────────────────────────────
BATCH_SIZE = 64
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LR_PATIENCE = 3       # reduce LR if val loss doesn't improve for N epochs
EARLY_STOP_PATIENCE = 6
NUM_WORKERS = 2
RANDOM_SEED = 42

# ── Image ─────────────────────────────────────────────────────────────────────
IMAGE_SIZE = 224      # EfficientNet-B0 standard input
# ImageNet normalisation — used because EfficientNet-B0 was pre-trained on ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 2. Load Data

In [ ]:
# ── Option A: mount Google Drive ──────────────────────────────────────────────
if DATA_SOURCE == "gdrive":
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path(GDRIVE_DATA_PATH)

# ── Option B: download from S3 ────────────────────────────────────────────────
else:
    import boto3
    from google.colab import userdata

    # store AWS keys as Colab secrets (🔑 icon in the left sidebar)
    os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
    os.environ["AWS_DEFAULT_REGION"]    = "eu-west-1"

    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Downloading from s3://{S3_BUCKET}/{S3_DATA_PREFIX}/...")
    !aws s3 sync s3://{S3_BUCKET}/{S3_DATA_PREFIX}/ {LOCAL_DATA_DIR}/
    DATA_DIR = LOCAL_DATA_DIR

# load class metadata saved by preprocess.py
with open(DATA_DIR / "metadata.json") as f:
    metadata = json.load(f)

CLASS_NAMES = metadata["class_names"]
NUM_CLASSES = metadata["num_classes"]
print(f"Classes: {NUM_CLASSES}")
print(f"First 5: {CLASS_NAMES[:5]}")

## 3. DataLoaders

Why `torchvision.datasets.ImageFolder`?  
It reads a directory structured as `split/class_name/*.jpg` — exactly what `preprocess.py` produces — and returns (tensor, label_index) pairs ready for DataLoader. We add augmentation only on the training split.

In [ ]:
# Training transforms: resize + augmentation + normalize
# Augmentation teaches the model to be robust to real-world variation
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Val/test transforms: only resize + normalize (no augmentation)
eval_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_dataset = ImageFolder(root=str(DATA_DIR / "train"), transform=train_transforms)
val_dataset   = ImageFolder(root=str(DATA_DIR / "val"),   transform=eval_transforms)
test_dataset  = ImageFolder(root=str(DATA_DIR / "test"),  transform=eval_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_dataset)} images | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# sanity check: class order from ImageFolder must match metadata
assert train_dataset.classes == CLASS_NAMES, \
    f"Class mismatch between ImageFolder and metadata.json!\n{train_dataset.classes[:3]} vs {CLASS_NAMES[:3]}"

## 4. Model — EfficientNet-B0

Transfer learning strategy:
- Load EfficientNet-B0 with **ImageNet pre-trained weights**
- **Freeze** all layers initially — only train the new classification head
- After a few epochs, **unfreeze** all layers for full fine-tuning

In [ ]:
def build_model(num_classes: int, freeze_backbone: bool = True) -> nn.Module:
    """
    Load EfficientNet-B0 with ImageNet weights and replace the
    classifier head for our 38-class problem.
    """
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # replace the classifier head
    # original: Linear(1280, 1000) for ImageNet
    # ours:     Linear(1280, 38)   for PlantVillage
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, num_classes),
    )

    return model.to(DEVICE)


model = build_model(NUM_CLASSES, freeze_backbone=True)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Total params:     {sum(p.numel() for p in model.parameters()):,}")

## 5. Training Loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += images.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    f1 = f1_score(all_labels, all_preds, average="weighted")
    return total_loss / total, correct / total, f1

## 6. MLflow Training Run

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run() as run:
    RUN_ID = run.info.run_id
    print(f"MLflow run ID: {RUN_ID}")

    # ── log hyperparameters ───────────────────────────────────────────────────
    mlflow.log_params({
        "model": "efficientnet_b0",
        "pretrained": "imagenet",
        "num_classes": NUM_CLASSES,
        "batch_size": BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "image_size": IMAGE_SIZE,
        "optimizer": "AdamW",
        "scheduler": "ReduceLROnPlateau",
        "early_stop_patience": EARLY_STOP_PATIENCE,
        "train_size": len(train_dataset),
        "val_size": len(val_dataset),
        "test_size": len(test_dataset),
        "leaf_aware_split": metadata.get("leaf_aware_split", False),
    })

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=LR_PATIENCE, factor=0.5, verbose=True
    )

    best_val_loss = float("inf")
    best_val_f1 = 0.0
    epochs_no_improve = 0
    unfreeze_done = False

    for epoch in range(1, NUM_EPOCHS + 1):
        # ── unfreeze backbone after epoch 3 ──────────────────────────────────
        if epoch == 4 and not unfreeze_done:
            print("Unfreezing backbone for full fine-tuning...")
            for param in model.parameters():
                param.requires_grad = True
            optimizer = optim.AdamW(
                model.parameters(),
                lr=LEARNING_RATE / 10,  # lower LR for fine-tuning
                weight_decay=WEIGHT_DECAY,
            )
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode="min", patience=LR_PATIENCE, factor=0.5, verbose=True
            )
            unfreeze_done = True

        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion)
        elapsed = time.time() - t0

        scheduler.step(val_loss)

        print(
            f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
            f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f} | "
            f"{elapsed:.1f}s"
        )

        # ── log metrics to MLflow ─────────────────────────────────────────────
        mlflow.log_metrics({
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1,
            "lr": optimizer.param_groups[0]["lr"],
        }, step=epoch)

        # ── save best model ───────────────────────────────────────────────────
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), "/content/best_model.pt")
        else:
            epochs_no_improve += 1

        # ── early stopping ────────────────────────────────────────────────────
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

    print(f"\nBest val loss: {best_val_loss:.4f} | Best val F1: {best_val_f1:.4f}")

## 7. Evaluate on Test Set

In [ ]:
# load best checkpoint and evaluate on held-out test set
model.load_state_dict(torch.load("/content/best_model.pt", map_location=DEVICE))

with mlflow.start_run(run_id=RUN_ID):
    test_loss, test_acc, test_f1 = evaluate(model, test_loader, criterion)
    print(f"Test accuracy: {test_acc:.4f} | Test F1: {test_f1:.4f}")

    mlflow.log_metrics({
        "test_loss": test_loss,
        "test_acc": test_acc,
        "test_f1": test_f1,
    })

    # full per-class classification report
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images.to(DEVICE))
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())

    report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES)
    print(report)

    # log report as artifact
    with open("/content/classification_report.txt", "w") as f:
        f.write(report)
    mlflow.log_artifact("/content/classification_report.txt")

## 8. Export to ONNX

Why ONNX?
- The inference service (FastAPI / Lambda) uses **ONNX Runtime** instead of PyTorch
- ONNX Runtime is ~100MB vs PyTorch ~2GB — critical for Lambda container size limits
- Same model, same predictions, much smaller footprint

In [ ]:
ONNX_PATH = "/content/model.onnx"

model.eval()
dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)

torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    opset_version=17,
    input_names=["image"],
    output_names=["logits"],
    dynamic_axes={
        "image":  {0: "batch_size"},
        "logits": {0: "batch_size"},
    },
)

print(f"ONNX model exported → {ONNX_PATH}")

# quick sanity check with onnxruntime
!pip install onnxruntime --quiet
import onnxruntime as ort

session = ort.InferenceSession(ONNX_PATH)
dummy_np = dummy_input.cpu().numpy()
onnx_out = session.run(["logits"], {"image": dummy_np})[0]
print(f"ONNX output shape: {onnx_out.shape}")  # should be (1, 38)

# verify outputs match PyTorch
with torch.no_grad():
    pt_out = model(dummy_input).cpu().numpy()
np.testing.assert_allclose(onnx_out, pt_out, rtol=1e-3, atol=1e-5)
print("PyTorch and ONNX outputs match ✓")

## 9. Log Model to MLflow and Register

In [ ]:
with mlflow.start_run(run_id=RUN_ID):

    # log ONNX model as artifact
    mlflow.log_artifact(ONNX_PATH, artifact_path="onnx")

    # log metadata.json so the inference service knows class names
    mlflow.log_artifact(str(DATA_DIR / "metadata.json"), artifact_path="onnx")

    # register the PyTorch model in the Model Registry
    # (registered separately from ONNX for experiment comparison)
    model_info = mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path="model",
        registered_model_name=MODEL_NAME,
    )

print(f"\nModel registered as: {MODEL_NAME}")
print(f"Run ID: {RUN_ID}")
print(f"\nNext steps:")
print(f"  1. Go to MLflow UI → Models → {MODEL_NAME}")
print(f"  2. Promote version to 'Staging', then 'Production'")
print(f"  3. Set RUN_ID={RUN_ID} in Lambda env vars or SSM Parameter Store")

## 10. Promote to Production (after manual review)

In [ ]:
# Run this cell only after reviewing the metrics and deciding to promote

client = mlflow.tracking.MlflowClient()

# get the latest version
latest_version = client.get_latest_versions(MODEL_NAME, stages=["None"])[0].version

# transition to Staging first
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=latest_version,
    stage="Staging",
)
print(f"Version {latest_version} → Staging")

# then to Production (only after validation)
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=latest_version,
    stage="Production",
    archive_existing_versions=True,  # archive previous production model
)
print(f"Version {latest_version} → Production")
print(f"\nexport RUN_ID={RUN_ID}")